# Understand Cost and Usage of Operations

When using LLMs for evaluation and test set generation, cost will be an important factor. Ragas provides you some tools to help you with that.

## Implement `TokenUsageParser`

By default Ragas does not calculate token usage for `evaluate()`, because
providers report it in different shapes. To get usage data, supply a
`TokenUsageParser`.

A `TokenUsageParser` is a function that takes the provider's raw completion
object and returns a `TokenUsage`. Ragas ships parsers for the common providers.

For example, here is the OpenAI parser applied to a real completion:

In [4]:
import os

os.environ["OPENAI_API_KEY"] = "your-api-key"

In [ ]:
from openai import OpenAI

# lets import a parser for OpenAI
from ragas.cost import get_token_usage_for_openai

client = OpenAI()
completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "hai there"}],
)

get_token_usage_for_openai(completion)

You can define your own or import parsers if they are defined. If you would like to suggest parser for LLM providers or contribute your own ones please check out this [issue](https://github.com/vibrantlabsai/ragas/issues/1151) 🙂.

You can use it for evaluations as so. Using example from [get started](../../../getstarted/evals.md) here.

In [6]:
from datasets import load_dataset

from ragas import EvaluationDataset
from ragas.metrics._aspect_critic import AspectCriticWithReference

dataset = load_dataset("vibrantlabsai/amnesty_qa", "english_v3")


eval_dataset = EvaluationDataset.from_hf_dataset(dataset["eval"])

metric = AspectCriticWithReference(
    name="answer_correctness",
    definition="is the response correct compared to reference",
)

Repo card metadata block was not found. Setting CardData to empty.


In [ ]:
from ragas import evaluate
from ragas.cost import get_token_usage_for_openai
from ragas.llms import llm_factory
from ragas.metrics import LLMContextRecall

evaluator_llm = llm_factory("gpt-4o", client=client)

results = evaluate(
    eval_dataset,
    metrics=[LLMContextRecall()],
    llm=evaluator_llm,
    token_usage_parser=get_token_usage_for_openai,
)

In [9]:
results.total_tokens()

TokenUsage(input_tokens=5463, output_tokens=355, model='')

You can compute the cost for each run by passing in the cost per token to `Result.total_cost()` function.

In this case GPT-4o costs $5 for 1M input tokens and $15 for 1M output tokens.

In [10]:
results.total_cost(cost_per_input_token=5 / 1e6, cost_per_output_token=15 / 1e6)

0.03264